## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/nanotube.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXPERIMENT: path follower")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
TRAIL_COLOR = [0.0, 1.0, 1.0, 1.0]
PATH_COLOR = [1.0, 1.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 0.0, 0.0, 1.0]
CHOSEN_COLOR = [0.0, 1.0, 0.0, 1.0]

In [ ]:
import numpy as np
import math

from nanover.trajectory import FrameData
from nanover.imd import ParticleInteraction
from nanover.jupyter import ImdAgent


def START_FOLLOWER():
    agent = FollowerAgent.from_runner(imd_runner)
    agent.start()


class FollowerAgent(ImdAgent):
    strength = 1
    count = 0

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        # oscillate scale between 0 and 1
        self.count += 1
        time = self.count / 30  # 30fps
        u = (time / 4) % 1
        scale = math.sin(u * math.pi) * 10

        # scale between crushing particles to point and pushing them away
        centroid = np.average(full_frame.particle_positions, axis=0)
        vecs = full_frame.particle_positions - centroid
        norm = np.linalg.norm(vecs, axis=1).reshape(-1, 1)
        unit = vecs / norm
        next = full_frame.particle_positions + unit * scale

        # update interactions
        with self.interactions as interactions:
            for i in range(full_frame.particle_count):
                interaction = ParticleInteraction(position=[int(x) for x in next[i]], particles=[i], interaction_type="spring", scale=self.strength, max_force=100)
                interactions.update_interaction(f"squeeze.{i}", interaction)

In [4]:
import numpy as np
from nanover.jupyter import Mode, SceneObjectsUtility

DRAWING_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)

PATH_POINTS = {}
NEXT_PATH_ID = 0
CURSOR_PATH_ID = {}


class DrawPathMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        global NEXT_PATH_ID

        if button == "primary":
            CURSOR_PATH_ID[key] = NEXT_PATH_ID
            NEXT_PATH_ID += 1
            PATH_POINTS[CURSOR_PATH_ID[key]] = []

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            CURSOR_PATH_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        if not key in CURSOR_PATH_ID:
            return

        path_id = CURSOR_PATH_ID[key]
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        PATH_POINTS[path_id].append(next_pos)

        DRAWING_OBJECTS.update_line(f"path.{path_id}", positions=PATH_POINTS[path_id], size=0.05, color=PATH_COLOR)


utilities.use_interaction_modes()
utilities.add_interaction_mode(DrawPathMode, "draw path")

In [5]:
DrawPathMode().on_cursor_updated(key="TEST", cursor={"position":[0, 0, 0]})

In [6]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode

TRAIL_OBJECTS = SceneObjectsUtility.from_runner(imd_runner)
TRAIL_POINTS = {}


class InteractionTrailMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        TRAIL_POINTS[key] = []

    def on_interaction_updated(self, *, key: str, interaction: ParticleInteraction):
        indexes = [int(index) for index in interaction.particles]
        positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
        centroid = [float(x) for x in np.mean(positions[indexes], axis=0)]
        TRAIL_POINTS[key].append(centroid)
        utilities.notify_all(f"{centroid}")
        utilities.objects.update_line(f"trail.{key}", positions=TRAIL_POINTS[key], size=0.05, color=TRAIL_COLOR)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        utilities.notify_all(f"FINISHED TRAIL\n{key}")

utilities.add_interaction_mode(InteractionTrailMode, "trails")

In [7]:
import numpy as np
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode


CURSOR_HOVERED = {}


class PathFollowingMode(Mode):
    PATH_ID = None
    PARTICLES = None

    def check(self):
        if self.PARTICLES is not None and self.PATH_ID is not None:
            utilities.notify_all(f"FOLLOWING!")
            # START_FOLLOWER

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        self.PARTICLES = [int(index) for index in interaction.particles]
        utilities.notify_all(f"SELECTED PARTICLES: {self.PARTICLES}")
        self.check()

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        if button == "primary":
            hovered = CURSOR_HOVERED.get(key, None)
            if hovered is not None:
                utilities.notify_all(f"SELECTED PATH {hovered}")
                self.PATH_ID = hovered
                DRAWING_OBJECTS.update_line(f"selected.path.{key}", positions=PATH_POINTS[hovered], size=0.15, color=CHOSEN_COLOR)
                self.check()

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        HOVERED = None
        HOVERED_ID = None

        for path_id, path in PATH_POINTS.items():
            for point in path:
                distance = np.linalg.norm(np.subtract(next_pos, point))
                if distance < 1:
                    HOVERED = path
                    HOVERED_ID = path_id
                    break

        CURSOR_HOVERED[key] = HOVERED_ID

        if HOVERED is None:
            DRAWING_OBJECTS.remove_line(f"hover.{key}")
        else:
            DRAWING_OBJECTS.update_line(f"hover.{key}", positions=HOVERED, size=0.1, color=HOVER_COLOR)

utilities.add_interaction_mode(PathFollowingMode, "follow")

In [8]:
PathFollowingMode().on_cursor_updated(key="TEST", cursor={"position":[0, 0, 0]})